In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        # print(os.path.join(dirname, filename))
        pass

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# =========================================
# INSTALL LLAMA.CPP
# =========================================

!git clone https://github.com/ggml-org/llama.cpp
%cd llama.cpp

!cmake -B build
!cmake --build build --config Release -j2

# =========================================
# CONVERT HF MODEL -> GGUF F16
# =========================================



Cloning into 'llama.cpp'...
remote: Enumerating objects: 93157, done.
remote: Counting objects: 100% (155/155), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 93157 (delta 97), reused 65 (delta 61), pack-reused 93002 (from 3)
Receiving objects: 100% (93157/93157), 386.55 MiB | 30.52 MiB/s, done.
Resolving deltas: 100% (66162/66162), done.
/kaggle/working/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identific

In [3]:
# CHANGE THIS PATH
MODEL_PATH = "/kaggle/input/notebooks/mdnomanbiswassibly/gemma-4-med-domain-fine-tuning/truthlayer-med-f16"

!python convert_hf_to_gguf.py \
    {MODEL_PATH} \
    --outfile /kaggle/working/truthlayer-med-f16.gguf \
    --outtype f16

INFO:hf-to-gguf:Loading model: truthlayer-med-f16

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`
INFO:hf-to-gguf:Model architecture: Gemma4ForConditionalGeneration

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model

In [4]:
# =========================================
# CLEAN MEMORY
# =========================================

import gc
gc.collect()

# =========================================
# CHECK GGUF SIZE
# =========================================

import os

gguf_path = "/kaggle/working/truthlayer-med-f16.gguf"

size_gb = os.path.getsize(gguf_path) / (1024**3)

print(f"F16 GGUF size: {size_gb:.2f} GB")

# =========================================
# QUANTIZE -> Q4_K_M
# =========================================

!./build/bin/llama-quantize \
    /kaggle/working/truthlayer-med-f16.gguf \
    /kaggle/working/truthlayer-med-q4_k_m.gguf \
    Q4_K_M

F16 GGUF size: 8.67 GB
llama_print_build_info: build = 9120 (fde69a360)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/kaggle/working/truthlayer-med-f16.gguf' to '/kaggle/working/truthlayer-med-q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 39 key-value pairs and 601 tensors from /kaggle/working/truthlayer-med-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma4
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Truthlayer Med F16
llama_model_loader: - kv   3:                         general.size_label str              = 4.6B
llama_model_loader: - kv   4:                         gemma4.block_count u32              = 35
l

In [5]:
# =========================================
# VERIFY OUTPUT
# =========================================

quant_path = "/kaggle/working/truthlayer-med-q4_k_m.gguf"

size_gb = os.path.getsize(quant_path) / (1024**3)

print(f"Q4_K_M GGUF size: {size_gb:.2f} GB")

# =========================================
# LIST OUTPUT FILES
# =========================================

for f in os.listdir("/kaggle/working"):
    if f.endswith(".gguf"):
        size = os.path.getsize(f"/kaggle/working/{f}") / (1024**3)
        print(f"{f} -> {size:.2f} GB")

print("\nDONE! Download the Q4_K_M GGUF from Kaggle Output.")

Q4_K_M GGUF size: 3.19 GB
truthlayer-med-q4_k_m.gguf -> 3.19 GB
truthlayer-med-f16.gguf -> 8.67 GB

DONE! Download the Q4_K_M GGUF from Kaggle Output.
